# Reviewer-response validation for DfD / DfR patent analysis (V4)

**Purpose:** Audit existing saved outputs without changing the original analysis, prepare *blinded* expert annotation files, compute honest held-out validation when manually judged labels arrive, and run parameter/indicator sensitivity checks. No automatic output is called expert validation.

**How to use:** Open in Google Colab; make sure the original project's `TOA_Recycle_Results` folder is on your Drive. Run cells in order. Initial audit and annotation templates run from saved parquet files; optional sensitivity cells require cached embedding files and potentially a GPU. All new files go to `TOA_Recycle_Results/REVIEWER_REVISION_AUDIT` (the original results are not overwritten). For detailed Lens search history and original-language patent records, human-supplied source documents are still needed. Do not substitute guessed search strings.

**Important:** Two reviewers should independently code `expert_A_random.csv` and `expert_B_random.csv` without seeing predictions. Resolve disagreements and save `adjudicated_random.csv` into `MANUAL_DIR`. The `gold_function` field is one primary function among the eight prototype names or `NONE`, or `UNSURE` (excluded). Use the comment field for multiple functions. The random sample is for population-style assessment; the stratified sample intentionally includes more semantic assignments and direct assignments, and the critical-cluster sample is for investigation and should **not** be used to claim corpus-level accuracy.

In [ ]:
from IPython import get_ipython
get_ipython().system("pip -q install pyarrow scikit-learn scipy sentence-transformers umap-learn hdbscan matplotlib")

from pathlib import Path
from collections import Counter
from datetime import datetime
import os, json, re, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.metrics import (classification_report, confusion_matrix,
                             cohen_kappa_score, precision_recall_fscore_support,
                             adjusted_rand_score)
from google.colab import drive

drive.mount('/content/drive')
PROJECT_DIR = Path('/content/drive/MyDrive/Academic/RecycleTech')
RESULTS_ROOT = PROJECT_DIR / 'TOA_Recycle_Results'
DATA = RESULTS_ROOT / 'data'
EMBED = RESULTS_ROOT / 'embeddings'
TABLE = RESULTS_ROOT / 'tables'
REPORT = RESULTS_ROOT / 'reports'
AUDIT = RESULTS_ROOT / 'REVIEWER_REVISION_AUDIT'
MANUAL_DIR = AUDIT / 'manual_annotations'
for d in (AUDIT, MANUAL_DIR, AUDIT/'sensitivity', AUDIT/'expert_templates'):
    d.mkdir(parents=True, exist_ok=True)
SEED = 42
N_RANDOM = 300
PATENT_MODEL = 'AI-Growth-Lab/PatentSBERTa'
RUN_SENTENCE_THRESHOLD_SWEEP = True  # Needs V4 sentence embedding .npy; model encodes only 8 functions
RUN_ROS_ANCHOR_SENSITIVITY = True     # Needs 3128 patent embeddings + PatentSBERTa
RUN_CLUSTERING_SENSITIVITY = True     # HDBSCAN on cached UMAP + optional new UMAPs
RUN_EXTRA_UMAP_FITS = False          # Set True for UMAP hyperparameter sensitivity (costlier)

if not DATA.exists():
    raise FileNotFoundError(f'Missing saved results at {DATA}. Update PROJECT_DIR, or copy TOA_Recycle_Results to Drive.')
print('Existing results:', RESULTS_ROOT)
print('New audit outputs:', AUDIT)

def need(path):
    if not Path(path).exists():
        raise FileNotFoundError(f'Required saved artifact missing: {path}. Run/resume original V4 notebook or provide this file.')
    return Path(path)

def save(df, name, area=AUDIT):
    target = area / name
    df.to_csv(target, index=False, encoding='utf-8-sig')
    print(f'Saved {target} ({len(df):,} rows)')
    return target

def z(x):
    x = np.asarray(x,dtype=float)
    sd = np.nanstd(x)
    return (x-np.nanmean(x))/sd if sd>0 else np.zeros_like(x)

FAM = need(DATA/'02_family_deduplicated.parquet')
ASSIGN = need(DATA/'11_evidence_gated_assignments_FINAL_V4.parquet')
LABELS = need(DATA/'11_evidence_gated_labels_FINAL_V4.parquet')
FUNC = need(DATA/'09_representative_functions.parquet')
METRICS = need(DATA/'13_cluster_metrics_FINAL_V4.parquet')
fam = pd.read_parquet(FAM)
assign = pd.read_parquet(ASSIGN)
labels = pd.read_parquet(LABELS)
func = pd.read_parquet(FUNC)
metrics = pd.read_parquet(METRICS)
assert fam['doc_idx'].is_unique and assign['doc_idx'].is_unique
assert fam['family_id'].is_unique, 'Review family deduplication before interpreting stats'
assert set(assign['doc_idx'])==set(fam['doc_idx'])
fnames = func.sort_values('function_rank')['prototype_function'].astype(str).tolist()
print('Families',len(fam),'Rows in V4 assignments',len(assign),'Functions',len(fnames))
print('Function labels:',fnames)
print('Published analysis uses title + abstract; original Chinese language provenance cannot be inferred from translation automatically.')

In [ ]:
# 1: Raw files, saved output inventory, source data quality, reproducibility
items=[]
for p in sorted(RESULTS_ROOT.rglob('*')):
    if p.is_file() and AUDIT not in p.parents:
        items.append({'relative_path':str(p.relative_to(RESULTS_ROOT)), 'bytes':p.stat().st_size,
                      'modified_local':datetime.fromtimestamp(p.stat().st_mtime).isoformat()})
save(pd.DataFrame(items),'00_saved_artifact_inventory.csv')
cols = []
for name,df in [('families',fam),('assignments',assign),('labels',labels),('functions',func),('cluster_metrics',metrics)]:
    for c in df.columns:
        cols.append({'table':name,'column':c,'dtype':str(df[c].dtype),'non_null':int(df[c].notna().sum())})
save(pd.DataFrame(cols),'00_saved_column_schema.csv')
patent_ids = fam[['family_id','doc_idx'] + [c for c in ['lens_id','title','jurisdiction','language','publication_date','earliest_priority_date'] if c in fam.columns]].copy()
save(patent_ids,'00_patent_family_identifiers.csv')

final = assign[assign['assigned'].fillna(False).astype(bool)].copy()
src_counts = final['function_evidence_source'].fillna('UNKNOWN').value_counts().to_dict()
summary = {'total_families':len(fam),'field_gated':int(assign['field_assigned'].sum()),
           'function_gated':int(assign['function_assigned'].sum()),
           'final_both_gates':len(final),'final_evidence_source':src_counts,
           'n_priority_all':int(metrics['key_cluster_all'].sum()) if 'key_cluster_all' in metrics else None,
           'n_primary':int(metrics['primary_key_cluster'].sum()) if 'primary_key_cluster' in metrics else None}
(AUDIT/'00_reproduced_counts.json').write_text(json.dumps(summary,indent=2,ensure_ascii=False),encoding='utf8')
print(json.dumps(summary,indent=2))
assert final['family_id'].nunique()==len(final), 'Duplicate final family memberships need inspection'
save(final,'01_all_final_assignments_with_evidence.csv')

# Important: this is a proxy for the language of the exported text, NOT evidence of original/translation language.
from collections import Counter
lang_cols = [c for c in fam if any(w in c.lower() for w in ['language','translation','jurisdiction','country'])]
print('Language/jurisdiction-related columns available:',lang_cols)
if 'jurisdiction' in fam:
    j = fam['jurisdiction'].fillna('UNKNOWN').astype(str)
    jtab = pd.DataFrame({'jurisdiction':j.value_counts().index,'families':j.value_counts().values})
    save(jtab,'01_jurisdiction_counts.csv')
    j2 = fam[['doc_idx','jurisdiction']].merge(assign[['doc_idx','assigned','function_assigned','field_assigned']],on='doc_idx')
    jurqa = j2.groupby('jurisdiction',dropna=False)[['assigned','function_assigned','field_assigned']].agg(['sum','mean','count'])
    jurqa.columns = ['_'.join(x) for x in jurqa.columns]
    save(jurqa.reset_index(),'01_jurisdiction_assignment_rates.csv')
print('EXTERNAL INPUT STILL REQUIRED: exact Lens Boolean query, search execution date, filters, original-language/translation metadata.')
# Data lineage: list raw columns that might document translation and retrieval (do not infer missing data).
rawp=DATA/'01_raw_normalized.parquet'
if rawp.exists():
    raw=pd.read_parquet(rawp)
    save(pd.DataFrame({'column':list(raw.columns),'n_non_null':[int(raw[x].notna().sum()) for x in raw.columns]}),
         '01_lens_export_source_columns.csv')
    print('Raw export columns suggestive of language, translation or search context:',
          [c for c in raw.columns if any(t in c.lower() for t in ['lang','translat','claim','priority','publication','jurisdiction'])])


In [ ]:
# 2: Extract actual selected functions, critical-cluster family IDs, and transparent field membership.
save(func,'02_selected_eight_function_prototypes.csv')
save(metrics,'02_all_opportunity_metrics_original.csv')

critical_ids = [41,100,75,76,16,102,81]
critical = final[final['label_id'].isin(critical_ids)].merge(
    fam[['doc_idx','abstract','cpc_classifications'] +
        [c for c in ['jurisdiction','publication_date'] if c in fam.columns]],
    on='doc_idx', how='left',suffixes=('','_family'))
save(critical,'02_critical_cluster_model_evidence_INTERNAL.csv')
print('Critical cluster IDs / family counts:',critical.groupby('label_id')['family_id'].nunique().to_dict())
for cid in [41,100]:
    x=critical[critical['label_id']==cid]
    print(f'Cluster {cid}:',len(x),'families; Lens IDs: ',x['lens_id'].astype(str).tolist())

# Recover CPC codes from actual family record, not semantic similarity.
def cpc_three(code):
    return re.findall(r'(?<![A-Z0-9])([A-HY]\d{2})(?=[A-Z0-9/\s;,:-]|$)',str(code).upper())
check = final.merge(fam[['doc_idx','cpc_classifications']],on='doc_idx',how='left')
check['cpc_seen_in_export'] = check.apply(
    lambda r: str(r['field_cpc_class']) in set(cpc_three(r['cpc_classifications'])),axis=1)
save(check[~check['cpc_seen_in_export']], '02_field_gate_disagreements_CHECK.csv')
print('Field-CPC mismatches under simplified regex (verify CPC parsing):',int((~check['cpc_seen_in_export']).sum()))

# Four-character subclass alternatives: counts, not a full rerun of the published field score.
subclass_rows=[]
for r in fam[['family_id','cpc_classifications']].itertuples(index=False):
    for c in set(re.findall(r'(?<![A-Z0-9])([A-HY]\d{2}[A-Z])',str(r.cpc_classifications).upper())):
        subclass_rows.append((r.family_id,c,c[:3]))
if subclass_rows:
    sc=pd.DataFrame(subclass_rows,columns=['family_id','cpc_subclass','cpc_class'])
    save(sc.groupby(['cpc_class','cpc_subclass']).family_id.nunique().reset_index(name='families')
             .sort_values('families',ascending=False),'02_cpc_class_vs_subclass_counts.csv')

In [ ]:
# 3: Create TWO independent, blind annotation sets. Do not replace gold labels with model predictions.
# Random sample: supports corpus-style validation if everyone is assessed, without post hoc filtering.
# Targeted cluster review: helps debug small/controversial clusters, not population-level accuracy.
N = min(N_RANDOM,len(fam))
random_rows = fam.sample(n=N,random_state=SEED).copy()
manual_extra = critical[['family_id']].drop_duplicates()
critical_rows = fam[fam['family_id'].isin(manual_extra['family_id'])].copy()

visible_cols = ['family_id','lens_id','title','abstract','cpc_classifications','jurisdiction','publication_date']
visible_cols = [c for c in visible_cols if c in fam.columns]

def blank_sheet(source):
    d=source[visible_cols].copy()
    # Labels must be entered by humans after reading the patent; no system predictions are present here.
    for c in ['gold_function','gold_recycling_relevant_0_1','gold_dfd_relevant_0_1',
              'gold_physically_disassemblable_0_1','gold_transfer_feasible_0_1',
              'gold_mechanism','reviewer_notes','reviewer_confidence_1_to_5']:
        d[c]=''
    return d

# Diagnostic sample enriched for the 318 semantic cases and 29 direct cases among final assignments.
# Report these subgroup diagnostics separately; this set is not a simple random sample.
final_sources=final[['family_id','function_evidence_source']].copy()
semantic_ids=final_sources.loc[final_sources.function_evidence_source.eq('semantic_sentence'),'family_id']
direct_ids=final_sources.loc[final_sources.function_evidence_source.eq('direct_parser'),'family_id']
strat_ids=pd.concat([
    semantic_ids.sample(n=min(130,len(semantic_ids)),random_state=SEED),
    direct_ids.sample(n=min(100,len(direct_ids)),random_state=SEED),
])
remaining=fam[~fam.family_id.isin(strat_ids)]
neg_pool=remaining[~remaining.family_id.isin(final.family_id)]
strat_ids=pd.concat([strat_ids,neg_pool.family_id.sample(n=min(90,len(neg_pool)),random_state=SEED)])
strat_rows=fam[fam.family_id.isin(strat_ids)].copy()

for name,rows in [('random',random_rows),('critical',critical_rows),('stratified',strat_rows)]:
    sheet=blank_sheet(rows)
    for annotator in ['A','B']:
        save(sheet, f'expert_{annotator}_{name}.csv', AUDIT/'expert_templates')
    pred=rows[['family_id','doc_idx']].merge(
        assign[['doc_idx','function','function_evidence_source','field_cpc_class',
                'label_id','assigned','function_similarity','function_margin','function_evidence_sentence']],
        on='doc_idx',how='left')
    save(pred,f'INTERNAL_predictions_{name}_do_not_show_to_raters.csv')

# Small blind AO pairs table for separate expert checking of semantic merges.
mapfile=DATA/'09_function_normalization_map.parquet'
if mapfile.exists():
    fm=pd.read_parquet(mapfile)
    print('Function normalization map columns:',list(fm))
    if {'action_object','prototype_function'}.issubset(fm):
        candidates=fm[['action_object','prototype_function']].dropna().drop_duplicates()
        sample=candidates.sample(n=min(150,len(candidates)),random_state=SEED)
        sample['expert_merge_correct_0_1']=''
        sample['expert_notes']=''
        save(sample,'03_function_merge_manual_annotation.csv',AUDIT/'expert_templates')

print('Expert instructions: primary label must match one of these eight functions EXACTLY, or NONE / UNSURE:')
print('\n'.join(f'- {x}' for x in fnames))
print('Complete A and B independently. Adjudicate disagreements to MANUAL_DIR/adjudicated_random.csv and adjudicated_critical.csv.')

### Instructions to domain experts

Code each patent on its own title/abstract and, for the disputed groups, verify the complete patent claims separately. Use the eight definitions provided in `02_selected_eight_function_prototypes.csv`. For each family record, enter one dominant `gold_function` label or `NONE`; use `UNSURE` only when it cannot be adjudicated. Mark explicit end-of-life recycling relevance separately from service/maintenance relevance and physical non-destructive separability. Do not see `INTERNAL_predictions...` until after the labeling files are complete. Two raters work independently; their disagreements require a third adjudication/consensus file. State clearly in the manuscript that one dominant function per patent is the adopted evaluation unit. If multilabel function attribution is needed, modify the codebook and classification scoring accordingly. No reviewer can be replaced by a model-generated label.

In [ ]:
# 4: Run independent assessment ONLY after manually adjudicated gold files exist.
# Never score the 254 silver standard as external expert ground truth.
def evaluate_gold(kind):
    p=MANUAL_DIR/f'adjudicated_{kind}.csv'
    if not p.exists():
        print('Pending expert adjudication:',p)
        return
    gold=pd.read_csv(p,dtype={'family_id':'string'})
    if 'gold_function' not in gold:
        print('Missing gold_function in',p);return
    gold['gold_function']=gold['gold_function'].astype(str).str.strip()
    valid=set(fnames)|{'NONE'}
    seen=gold[gold['gold_function'].isin(valid)].copy()
    print(kind,': manually valid labels =',len(seen),'out of',len(gold))
    if len(seen)<30:
        print('Not enough completed expert judgments for a useful evaluation.');return
    assert seen['family_id'].is_unique
    pred=seen[['family_id','gold_function']].merge(
        assign[['family_id','function','function_evidence_source','assigned','field_assigned','function_assigned','function_similarity','function_margin']],
        on='family_id',how='left',validate='one_to_one')
    pred=pred.merge(fam[['family_id'] + (["jurisdiction"] if "jurisdiction" in fam.columns else [])],
                    on='family_id',how='left',validate='one_to_one')
    pred['predicted_function']=pred['function'].fillna('NONE')
    assert pred['predicted_function'].isin(valid).all(), 'Unexpected prototype name; check mapping before scoring'
    order=fnames+['NONE']
    report=classification_report(pred['gold_function'],pred['predicted_function'],
                                 labels=order,output_dict=True,zero_division=0)
    save(pd.DataFrame(report).T.reset_index(names='label'),f'04_{kind}_expert_function_precision_recall_f1.csv')
    cm=confusion_matrix(pred['gold_function'],pred['predicted_function'],labels=order)
    save(pd.DataFrame(cm,index=order,columns=order).reset_index(names='gold'),f'04_{kind}_confusion_matrix.csv')
    save(pred,f'04_{kind}_expert_pred_vs_gold.csv')
    print('Accuracy',round((pred['gold_function']==pred['predicted_function']).mean(),4))
    print('Macro F1',round(report['macro avg']['f1-score'],4))
    if 'jurisdiction' in pred.columns:
        pred['CN_vs_nonCN']=np.where(pred['jurisdiction'].astype(str).eq('CN'),'CN','nonCN')
        jrates=(pred.assign(is_correct=pred['gold_function'].eq(pred['predicted_function']))
                .groupby('CN_vs_nonCN').agg(n=('family_id','size'),accuracy=('is_correct','mean')).reset_index())
        save(jrates,f'04_{kind}_CN_nonCN_expert_accuracy.csv')
        print('Jurisdiction results are conditional on the annotated sample; small nonCN sizes warrant caution.')
    print('NOTE: critical (purposively sampled) and random (probability sample) estimates are not interchangeable.')
    for source,g in pred.groupby(pred['function_evidence_source'].fillna('unassigned')):
        print(source, 'n=',len(g),'accuracy=',round((g['gold_function']==g['predicted_function']).mean(),3))
    if {'gold_recycling_relevant_0_1'}.issubset(seen):
        save(seen[['family_id','gold_function','gold_recycling_relevant_0_1']],f'04_{kind}_recycling_judgments.csv')

for kind in ['random','critical','stratified']:
    evaluate_gold(kind)

# Optional inter-rater agreement on independently completed files (use exact shared family IDs).
for kind in ['random','critical','stratified']:
    pa,pb=[MANUAL_DIR/f'expert_{who}_{kind}.csv' for who in ['A','B']]
    if pa.exists() and pb.exists():
        a=pd.read_csv(pa)[['family_id','gold_function']].rename(columns={'gold_function':'A'})
        b=pd.read_csv(pb)[['family_id','gold_function']].rename(columns={'gold_function':'B'})
        both=a.merge(b,on='family_id',validate='one_to_one')
        both=both[both['A'].isin(fnames+['NONE'])&both['B'].isin(fnames+['NONE'])]
        if len(both)>1:
            print(kind,'inter-rater Cohen kappa:',cohen_kappa_score(both['A'],both['B']))
            save(both[both['A']!=both['B']],f'04_{kind}_expert_disagreements_for_adjudication.csv')

In [ ]:
# 4b: Audit manually reviewed semantic merges and extract function-parser error examples.
review_file = MANUAL_DIR/'expert_function_merge.csv'
if review_file.exists():
    m = pd.read_csv(review_file)
    if 'expert_merge_correct_0_1' in m:
        valid = pd.to_numeric(m['expert_merge_correct_0_1'],errors='coerce').dropna()
        if len(valid):
            assert valid.isin([0,1]).all(), 'Expected binary 0/1 manual merge correctness.'
            save(pd.DataFrame([{'n_expert_checked_pairs':len(valid),'fraction_correct_on_sample':float(valid.mean())}]),
                 '04b_manual_semantic_merge_audit.csv')
        else: print('Manual merging review file exists but has no assessed pairs yet.')
else: print('Optional expert merge check pending. Fill 03_function_merge_manual_annotation.csv and save as',review_file)

rawfun = DATA/'08_raw_mechanism_functions.parquet'
if rawfun.exists():
    rf=pd.read_parquet(rawfun)
    keep=[c for c in ['family_id','lens_id','doc_idx','sentence','action_object','mechanism'] if c in rf.columns]
    if keep:
        sample=rf[keep].drop_duplicates().sample(n=min(200,len(rf[keep].drop_duplicates())),random_state=SEED).copy()
        for c in ['expert_action_object_correct_0_1','expert_generic_expression_0_1','expert_notes']:
            sample[c]=''
        save(sample,'04b_parser_good_bad_examples_for_expert.csv',AUDIT/'expert_templates')
        print('Raw function examples exported for human assessment; accuracy cannot be computed before manual coding.')


In [ ]:
# 5: Recompute sentence -> function similarity and sweep similarity/margin thresholds.
# This changes ONLY assignment decisions; it is not a full recomputation of TV, TN, ROS or reference fields.
if RUN_SENTENCE_THRESHOLD_SWEEP:
    sf=EMBED/'11_all_patent_sentence_embeddings_FINAL_V4.npy'
    sentence_file=DATA/'11_all_patent_sentences_FINAL_V4.parquet'
    if not sf.exists() or not sentence_file.exists():
        print('Skipping threshold sweep: missing original sentence-level V4 embeddings/table.')
    else:
        from sentence_transformers import SentenceTransformer
        import torch
        model=SentenceTransformer(PATENT_MODEL,device='cuda' if torch.cuda.is_available() else 'cpu')
        fv=model.encode(fnames,normalize_embeddings=True,convert_to_numpy=True,show_progress_bar=False).astype('float32')
        sentence_embedding=np.load(sf,mmap_mode='r')
        sdf=pd.read_parquet(sentence_file)
        assert len(sdf)==len(sentence_embedding)
        assert sdf['sentence_id'].is_unique
        assert sdf['sentence_id'].min()==0 and sdf['sentence_id'].max()==len(sdf)-1
        sims=np.asarray(sentence_embedding,dtype='float32') @ fv.T
        doc_ids=sdf['doc_idx'].to_numpy(dtype=int)
        score=np.full((len(fam),len(fnames)),-np.inf,dtype='float32')
        for j in range(len(fnames)):
            np.maximum.at(score[:,j],doc_ids,sims[:,j])
        order=np.argsort(score,axis=1)
        best=order[:,-1]
        sim1=score[np.arange(len(score)),best]
        sim2=score[np.arange(len(score)),order[:,-2]]
        margin=sim1-sim2
        assert sorted(fam['doc_idx'].astype(int))==list(range(len(fam)))
        core=assign.sort_values('doc_idx')
        direct=core['function_evidence_source'].eq('direct_parser').to_numpy()
        field=core['field_assigned'].astype(bool).to_numpy()
        oldlabel=core['function'].fillna('NONE').to_numpy(dtype=object)
        grid=[]
        for t in [0.375,0.40,0.45,0.50,0.55,0.60]:
            for m in [0,.01,.02,.03,.05]:
                passed=(sim1>=t)&(margin>=m)
                accepted=direct | (~direct & passed)
                currentfn=np.where(direct,oldlabel,np.where(passed,np.array(fnames,dtype=object)[best],'NONE'))
                final_now=field & accepted
                selected=core.loc[final_now,['family_id','field_cpc_class','doc_idx']].copy()
                selected['function_candidate']=currentfn[final_now]
                grid.append({'sim_threshold':t,'margin_threshold':m,'function_accepted':int(accepted.sum()),
                             'final_field_function':int(final_now.sum()),
                             'semantic_in_final':int((final_now & (~direct)).sum()),
                             'function_agreement_with_published':round(float(np.mean(currentfn==oldlabel)),6),
                             'final_family_jaccard':round(float((final_now & core['assigned'].to_numpy(dtype=bool)).sum() / max(1,(final_now | core['assigned'].to_numpy(dtype=bool)).sum())),6)})
        sweep=pd.DataFrame(grid)
        save(sweep,'05_function_gate_threshold_and_margin_sensitivity.csv',AUDIT/'sensitivity')
        save(pd.DataFrame({'doc_idx':core['doc_idx'],'family_id':core['family_id'],
                           'top_function':[fnames[j] for j in best],
                           'top_similarity':sim1,'top2_margin':margin,'is_direct_parser':direct,
                           'original_predicted_function':oldlabel,'field_assigned':field}),
             '05_function_semantic_scores_for_manual_check.csv',AUDIT/'sensitivity')
        x=sweep[(np.isclose(sweep.sim_threshold,.375))&(np.isclose(sweep.margin_threshold,0))].iloc[0]
        print('Reference thresholds .375 and .000 -> estimated final families:',int(x.final_field_function))
        if int(x.final_field_function)!=int(core['assigned'].sum()):
            print('WARNING: recalculated baseline mismatches original; investigate model version, cached embeddings and mapping before citing sensitivity numbers.')
        print(sweep[['sim_threshold','margin_threshold','final_field_function','final_family_jaccard']].to_string(index=False))

In [ ]:
# 6: Score sensitivity and metric correlations with existing V4 outputs.
# Original TV definition is reproduced from the stored raw components before testing alternatives.
from scipy.stats import spearmanr
req=['cluster_id','size_factor','median_age_normalized_forward_citations',
     'median_simple_family_size','TR_raw_similarity','ROS_raw_mean','TV','TR','TN','ROS']
missing=[c for c in req if c not in metrics]
if missing:
    print('Missing needed columns for score sensitivity:',missing)
else:
    m=metrics.copy().sort_values('cluster_id').reset_index(drop=True)
    c=z(m['median_age_normalized_forward_citations'])
    f=z(m['median_simple_family_size'])
    tv_base=(c+f)*m['size_factor'].to_numpy()/2
    err=np.nanmax(np.abs(tv_base-m['TV'].to_numpy(dtype=float)))
    print('Recomputed original TV maximum difference:',err)
    if err>1e-5: print('WARNING TV reproduction failed; review saved pipeline/version before citation.')
    variants=[]
    for w in [0.25,0.5,0.75]:
        for size in [True,False]:
            value=(w*c+(1-w)*f)*(m['size_factor'].to_numpy() if size else 1)
            screen=(value>0)&(m['TR'].to_numpy(dtype=float)>0)
            for ix,r in m.iterrows():
                variants.append({'cluster_id':r['cluster_id'],'cit_weight':w,'use_size_factor':size,
                                 'TV_variant':value[ix],'pass_TV_TR':bool(screen[ix]),
                                 'baseline_pass':bool(r.get('key_cluster_all',False)),
                                 'is_primary_n10':bool(screen[ix] and r['cluster_size']>=10)})
    scoregrid=pd.DataFrame(variants)
    save(scoregrid,'06_tv_weight_and_support_sensitivity.csv',AUDIT/'sensitivity')
    metric_cols=['TV','TR','TN','ROS','cluster_size']
    corr=m[metric_cols].corr(method='spearman')
    save(corr.reset_index(names='metric'),'06_metric_spearman_correlations.csv',AUDIT/'sensitivity')
    for cid in [100,41]:
        print('Cluster',cid,'TV variants:')
        print(scoregrid[scoregrid['cluster_id']==cid][['cit_weight','use_size_factor','TV_variant','pass_TV_TR']].to_string(index=False))

# TN upper-quartile + age weighting sensitivity from cached cluster/sentence similarities.
tn_sim=EMBED/'13_tn_cluster_sentence_similarity_FINAL_V4.npy'
field_corpus=DATA/'12_field_interest_sentences.parquet'
if tn_sim.exists() and field_corpus.exists():
    mat=np.load(tn_sim,mmap_mode='r')
    sent=pd.read_parquet(field_corpus)
    original_cluster_ids=np.sort(metrics.cluster_id.astype(int).unique())
    if mat.shape==(len(original_cluster_ids),len(sent)):
        age=((pd.Timestamp('2026-09-13')-pd.to_datetime(sent['publication_date'],errors='coerce')).dt.days/365.25).clip(lower=1).fillna(1).to_numpy()
        trows=[]
        for quantile in [0.65,0.75,0.85]:
            q=np.quantile(mat,quantile)
            for age_power in [0,.5,1]:
                wt=np.power(age,-age_power)
                overlap=(np.asarray(mat)>=q)@wt
                tn_alt=-z(overlap)
                for cid,tv in zip(original_cluster_ids,tn_alt):
                    trows.append({'cluster_id':int(cid),'overlap_quantile':quantile,'age_power':age_power,'TN_variant':tv})
        save(pd.DataFrame(trows),'06_tn_quantile_age_weight_sensitivity.csv',AUDIT/'sensitivity')
    else: print('TN cache dimension mismatch; cannot interpret without re-alignment:',mat.shape,len(original_cluster_ids),len(sent))
else: print('No TN similarity matrix or field-of-interest corpus; TN sensitivity deferred.')

In [ ]:
# 6b: Age-floor, country/field normalization and screening-rule sensitivity.
# Does not measure real economic value. It probes score dependence on the selected formula.
if {'cited_by_patent_count','simple_family_size','family_earliest_publication_date'}.issubset(fam.columns):
    work = fam.merge(assign[['doc_idx','assigned','label_id','field_cpc_class']],on='doc_idx',validate='one_to_one')
    work=work[work.assigned.astype(bool)].copy()
    analysis_date=pd.Timestamp('2026-09-13')
    yrs=((analysis_date-pd.to_datetime(work.family_earliest_publication_date,errors='coerce')).dt.days/365.25).fillna(1)
    cites=pd.to_numeric(work.cited_by_patent_count,errors='coerce').fillna(0).clip(lower=0)
    sizes=pd.to_numeric(work.simple_family_size,errors='coerce').fillna(1).clip(lower=1)
    vrows=[]
    for floor in [0.5,1.0,2.0]:
        base_rate=cites/yrs.clip(lower=floor)
        for adjustment in ['none','within_CPC','within_jurisdiction']:
            wk=work[['label_id','field_cpc_class']].copy()
            wk['citation_rate']=base_rate.to_numpy()
            wk['patent_family_size']=sizes.to_numpy()
            if adjustment=='within_CPC':
                grp=work['field_cpc_class'].fillna('UNKNOWN')
                rate_adj=pd.DataFrame({'group':grp,'val':np.log1p(base_rate)}).groupby('group')['val'].transform(
                    lambda s: (s-s.mean())/s.std(ddof=0) if s.std(ddof=0)>0 else s*0)
                wk['citation_rate']=rate_adj.to_numpy()
            if adjustment=='within_jurisdiction':
                if 'jurisdiction' not in work:continue
                grp=work['jurisdiction'].fillna('UNKNOWN')
                rate_adj=pd.DataFrame({'group':grp,'val':np.log1p(base_rate)}).groupby('group')['val'].transform(
                    lambda s: (s-s.mean())/s.std(ddof=0) if s.std(ddof=0)>0 else s*0)
                wk['citation_rate']=rate_adj.to_numpy()
            med=wk.groupby('label_id').agg(median_rate=('citation_rate','median'),median_size=('patent_family_size','median'),n=('citation_rate','size'))
            tv=((z(med.median_rate)+z(med.median_size))/2)*(1+np.log(med.n.to_numpy()))
            for cid,value in zip(med.index,tv):
                vrows.append({'cluster_id':int(cid),'age_floor_years':floor,'citation_adjustment':adjustment,
                              'TV_variant':float(value)})
    variance=pd.DataFrame(vrows).merge(metrics[['cluster_id','TR','ROS','cluster_size','TV']],on='cluster_id',how='left')
    variance['passes_TV_TR']=(variance['TV_variant']>0)&(variance['TR']>0)
    variance['passes_with_ROS']=variance['passes_TV_TR']&(variance['ROS']>0)
    variance['passes_without_Y10_singletons']=variance['passes_TV_TR']&(variance['cluster_size']>=2)
    if 'cpc_class' in metrics:
        variance=variance.merge(metrics[['cluster_id','cpc_class']],on='cluster_id',how='left')
        variance['passes_without_Y10_singletons'] &= variance['cpc_class'].ne('Y10')
    save(variance,'06b_tv_age_country_field_and_screen_sensitivity.csv',AUDIT/'sensitivity')
    print('Alternative screen counts (score-based exploratory analysis only):')
    print(variance.groupby(['age_floor_years','citation_adjustment'])[['passes_TV_TR','passes_with_ROS','passes_without_Y10_singletons']].sum().to_string())
else:
    print('Cannot rerun TV age/country/field controls: family citation, publication date, or size variables missing.')


In [ ]:
# 7: ROS sensitivity: original anchor sets from the active V2/V4 implementation.
# These are NOT invented judgments; the alternatives are deliberately specified sensitivity scenarios.
if RUN_ROS_ANCHOR_SENSITIVITY:
    patent_emb=EMBED/'patent_embeddings.npy'
    if not patent_emb.exists():
        print('No patent embeddings for ROS; skipping.')
    else:
        from sentence_transformers import SentenceTransformer
        import torch
        if 'model' not in globals():
            model=SentenceTransformer(PATENT_MODEL,device='cuda' if torch.cuda.is_available() else 'cpu')
        patent_matrix=np.asarray(np.load(patent_emb,mmap_mode='r'),dtype='float32')
        recycling_original=[
          'end-of-life material recovery for recycling','product recyclability',
          'component separation for recycling','closed-loop material recycling',
          'recovery of recyclable materials from products','end-of-life disassembly for material recovery',
          'sorting and separation of recyclable product materials']
        service_original=[
          'maintenance and servicing access','repair of a product',
          'replacement of a failed component','serviceability and maintenance disassembly',
          'assembly access for repair']
        scenarios={
          'original_7_top3_vs_service_max':(recycling_original,service_original,3,'max'),
          'original_7_top1_vs_service_max':(recycling_original,service_original,1,'max'),
          'original_7_mean_vs_service_mean':(recycling_original,service_original,0,'mean'),
          'short_recycling_4_top3':(recycling_original[:4],service_original,3,'max'),
          'end_of_life_4_top3':([recycling_original[i] for i in [0,2,4,5]],service_original,3,'max'),
        }
        unique_texts=list(dict.fromkeys(x for pos,neg,k,agg in scenarios.values() for x in pos+neg))
        e=model.encode(unique_texts,normalize_embeddings=True,convert_to_numpy=True,show_progress_bar=False).astype('float32')
        v={t:e[i] for i,t in enumerate(unique_texts)}
        group_ids=assign.sort_values('doc_idx')
        kept=group_ids[group_ids['assigned'].astype(bool)][['doc_idx','label_id']]
        original_label_ids=np.sort(metrics.cluster_id.astype(int).unique())
        rec=[]
        for label,(pos,neg,k,service_agg) in scenarios.items():
            ps=patent_matrix@np.stack([v[x] for x in pos]).T
            ns=patent_matrix@np.stack([v[x] for x in neg]).T
            pos_agg=np.sort(ps,axis=1)[:,-min(k,len(pos)):].mean(axis=1) if k>0 else ps.mean(axis=1)
            neg_agg=ns.max(axis=1) if service_agg=='max' else ns.mean(axis=1)
            tmp=kept.assign(ros_raw=(pos_agg-neg_agg)[kept['doc_idx'].astype(int).to_numpy()])
            means=tmp.groupby('label_id')['ros_raw'].mean()
            standardized=z(means.to_numpy())
            for cid,val in zip(means.index,standardized):
                rec.append({'cluster_id':int(cid),'scenario':label,'ROS_variant':float(val),'ROS_raw_mean':float(means.loc[cid])})
        save(pd.DataFrame(rec),'07_ros_anchors_and_aggregation_sensitivity.csv',AUDIT/'sensitivity')
        (AUDIT/'07_original_and_alternative_anchors.json').write_text(json.dumps(
            {n:{'recycling':a,'service':b,'top_k':k,'service_aggregation':ag} for n,(a,b,k,ag) in scenarios.items()},
            ensure_ascii=False,indent=2),encoding='utf8')
        print('ROS scenarios recomputed; compare to published scores before citing.')

In [ ]:
# 8: UMAP / HDBSCAN parameter sensitivity on cached embeddings.
# This measures BASE cluster stability only; it does NOT prove stability of downstream 347 assignments.
if RUN_CLUSTERING_SENSITIVITY:
    import hdbscan
    base_emb=EMBED/'umap_5d.npy'
    base_labels=DATA/'06_base_cluster_assignments.parquet'
    if not base_emb.exists() or not base_labels.exists():
        print('Clustering sensitivity skipped; need 04 UMAP5 and 06 base assignments.')
    else:
        X=np.asarray(np.load(base_emb,mmap_mode='r'))
        b=pd.read_parquet(base_labels).sort_values('doc_idx')
        print('Base cluster columns',list(b))
        label_col=next((c for c in ['base_topic','topic','cluster','cluster_id'] if c in b.columns),None)
        reference=b[label_col].to_numpy() if label_col and len(b)==len(X) else None
        records=[]
        for mc in [10,15,20,25,30,50]:
            for ms in [None,5,10]:
                if ms is not None and ms>mc:continue
                fit=hdbscan.HDBSCAN(min_cluster_size=mc,min_samples=ms,metric='euclidean',cluster_selection_method='eom')
                result=fit.fit_predict(X)
                valid=result>=0
                records.append({'umap_neighbors':15,'umap_components':5,'hdbscan_min_cluster_size':mc,
                                'hdbscan_min_samples':str(ms),'n_clusters':len(set(result[valid])),
                                'noise_frac':float((~valid).mean()),
                                'ARI_vs_published_base':adjusted_rand_score(reference,result) if reference is not None else np.nan})
        save(pd.DataFrame(records),'08_hdbscan_base_cluster_stability.csv',AUDIT/'sensitivity')

        if RUN_EXTRA_UMAP_FITS:
            from umap import UMAP
            embedfull=EMBED/'patent_embeddings.npy'
            if embedfull.exists():
                P=np.asarray(np.load(embedfull,mmap_mode='r'),dtype='float32')
                records2=[]
                for nn in [10,15,30]:
                    for nd in [5,10]:
                        for md in [0,0.1]:
                            X2=UMAP(n_neighbors=nn,n_components=nd,min_dist=md,metric='cosine',random_state=SEED,low_memory=True).fit_transform(P)
                            output=hdbscan.HDBSCAN(min_cluster_size=15,min_samples=None,metric='euclidean').fit_predict(X2)
                            mask=output>=0
                            records2.append({'n_neighbors':nn,'n_components':nd,'min_dist':md,
                                             'n_clusters':len(set(output[mask])), 'noise_frac':float((~mask).mean()),
                                             'ARI_vs_published_base':adjusted_rand_score(reference,output) if reference is not None else np.nan})
                save(pd.DataFrame(records2),'08_umap_hdbscan_base_cluster_stability.csv',AUDIT/'sensitivity')
    print('For full downstream field/functional opportunity stability, rerun reference-field extraction and V4 stages 07-15 under each setting with separate output directories.')

In [ ]:
# 9: Paired comparison with Kim et al. (2026), ONLY if faithful baseline outputs are supplied.
# This cell does not generate or invent a Kim baseline; it evaluates one if the user supplies it.
basefile=MANUAL_DIR/'kim2026_same_corpus_predictions.csv'  # Required cols: family_id,predicted_function
expertfile=MANUAL_DIR/'adjudicated_random.csv'
if basefile.exists() and expertfile.exists():
    base=pd.read_csv(basefile,dtype={'family_id':'string'})
    gold=pd.read_csv(expertfile,dtype={'family_id':'string'})
    assert {'family_id','predicted_function'}.issubset(base.columns)
    assert {'family_id','gold_function'}.issubset(gold.columns)
    eval_rows=gold[['family_id','gold_function']].merge(
        base[['family_id','predicted_function']].rename(columns={'predicted_function':'kim_pred'}),
        on='family_id',validate='one_to_one').merge(
        assign[['family_id','function']].rename(columns={'function':'v4_pred'}),
        on='family_id',validate='one_to_one')
    eval_rows['v4_pred']=eval_rows['v4_pred'].fillna('NONE')
    eval_rows['kim_pred']=eval_rows['kim_pred'].fillna('NONE')
    valid=set(fnames)|{'NONE'}
    eval_rows=eval_rows[eval_rows.gold_function.isin(valid) & eval_rows.kim_pred.isin(valid)]
    summary=[]
    for typ, col in [('Kim2026_faithful_baseline','kim_pred'),('V4_evidence_gated','v4_pred')]:
        pr,rc,f1,sup=precision_recall_fscore_support(eval_rows.gold_function,eval_rows[col],
                                                     labels=fnames+['NONE'],average='macro',zero_division=0)
        summary.append({'method':typ,'same_n':len(eval_rows),'accuracy':float((eval_rows.gold_function==eval_rows[col]).mean()),
                        'macro_precision':pr,'macro_recall':rc,'macro_f1':f1})
    save(pd.DataFrame(summary),'09_same_corpus_Kim_vs_V4_EXPERT_evaluation.csv')
    save(eval_rows,'09_paired_Kim_vs_V4_patents.csv')
else:
    print('No valid Kim et al. 2026 comparison yet. Required:',basefile,expertfile)
    print('Do not describe earlier V1/V2/V3 code as Kim baseline without verifying that it faithfully implements the published algorithm.')

In [ ]:
# 9: Export a compact collection of outputs for returning to coauthors / revision workflow.
import zipfile
zip_file = AUDIT/'SEND_ME_REVIEWER_REVISION_AUDIT.zip'
with zipfile.ZipFile(zip_file,'w',zipfile.ZIP_DEFLATED) as zf:
    for fp in sorted(AUDIT.rglob('*')):
        if not fp.is_file() or fp==zip_file:continue
        # Never send entire patent embeddings or raw giant model artifacts in an audit ZIP.
        if fp.suffix.lower() in {'.csv','.json','.txt','.md'}:
            zf.write(fp,fp.relative_to(AUDIT))
print('Download this completed audit ZIP to share:',zip_file)
print('External requirements remain: exact Lens search expression/date/filters; originals of controversial patents; Kim et al algorithm; blind expert ratings; design case measurements if claims are quantitative.')